# Wheat Strategy Backtest Research

Standalone wheat notebook. Wheat is researched as an SRW/HRW relative-value sleeve using generic A/B diagnostics, fixed pair tests, walk-forward, and named-period check.

In [1]:
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from grain_futures_strategy import (
    backtest_positions_with_costs,
    build_feature_panels,
    load_train_set,
    rolling_zscore,
    split_performance,
)
from research_config import DEFAULT_MARGIN_PER_LOT, REGIME_PERIODS, SPLIT_DATE

DATA_DIR = "train_set"
OOS_START = pd.Timestamp(SPLIT_DATE)
TRAIN_END = pd.Timestamp("2016-01-01")
DD_CAPITAL_USD = 10_000.0
TRADE_COST_PER_LOT = 8.75
HOLDING_COST_RATE = 0.05

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

## 1. Load Data And Confirm Cargill Inputs

In [2]:
data = load_train_set(DATA_DIR)
feature_panels, futures_pnl_all = build_feature_panels(data)
trading_index = futures_pnl_all.index
futures_pnl = futures_pnl_all[["WHEAT_SRW", "WHEAT_HRW"]]
srw_panel = feature_panels["WHEAT_SRW"].reindex(trading_index).fillna(0.0)
hrw_panel = feature_panels["WHEAT_HRW"].reindex(trading_index).fillna(0.0)

display(pd.DataFrame([
    {"commodity": c, "rows": len(trading_index), "start": trading_index.min(), "end": trading_index.max(), "has_cargill_inputs": {"cgl_inventory_change", "crush_surprise", "crush_utilization"}.issubset(feature_panels[c].columns)}
    for c in ["WHEAT_SRW", "WHEAT_HRW"]
]))

,commodity,rows,start,end,has_cargill_inputs
0,WHEAT_SRW,2866,2010-01-04,2020-12-31,True
1,WHEAT_HRW,2866,2010-01-04,2020-12-31,True


## 2. Standalone Helpers

In [3]:
def clean_signal(series, target_index):
    return (
        pd.Series(series, index=target_index)
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .clip(-5.0, 5.0)
    )


def family_average(families, target_index):
    signals = []
    for members in families.values():
        signals.extend(members)
    return clean_signal(pd.concat(signals, axis=1).mean(axis=1), target_index)


def equal_family(families, target_index):
    family_signals = [pd.concat(members, axis=1).mean(axis=1) for members in families.values()]
    return clean_signal(pd.concat(family_signals, axis=1).mean(axis=1), target_index)


def positions_from_signal(signal, pnl, commodity, target_daily_vol=75.0, max_abs_lot=0.50):
    signal = clean_signal(signal, pnl.index)
    signal = pd.Series(np.tanh(signal / 2.0), index=pnl.index)
    signal = signal.ewm(halflife=3.0, adjust=False, min_periods=1).mean()
    signal[signal.abs() < 0.05] = 0.0
    vol = pnl[commodity].rolling(60, min_periods=20).std().shift(1).replace(0.0, np.nan)
    lots = (signal * float(target_daily_vol) / vol).clip(-float(max_abs_lot), float(max_abs_lot)).fillna(0.0)
    return pd.DataFrame({commodity: lots}, index=pnl.index)


def backtest_signal(signal, pnl, commodity):
    return backtest_positions_with_costs(
        positions_from_signal(signal, pnl, commodity),
        pnl,
        trade_cost_per_lot=TRADE_COST_PER_LOT,
        holding_cost_rate=HOLDING_COST_RATE,
        margin_per_lot=DEFAULT_MARGIN_PER_LOT,
    )[0]


def active_metrics(bt, mask=None):
    sample_bt = bt if mask is None else bt.loc[mask]
    active = sample_bt["held_gross_exposure"] > 1.0e-12
    pnl = sample_bt.loc[active, "net_pnl"].dropna()
    if len(pnl) == 0:
        return {"days": np.nan, "total_pnl": np.nan, "sharpe": np.nan, "max_drawdown": np.nan, "hit_rate": np.nan, "turnover": np.nan}
    cum = pnl.cumsum()
    dd = cum - cum.cummax()
    vol = pnl.std()
    return {
        "days": float(len(pnl)),
        "total_pnl": float(pnl.sum()),
        "sharpe": np.nan if vol == 0.0 else float(pnl.mean() / vol * np.sqrt(252.0)),
        "max_drawdown": float(dd.min()),
        "hit_rate": float((pnl > 0.0).mean()),
        "turnover": float(sample_bt.loc[pnl.index, "turnover"].mean()),
    }


def metric_row(label, bt):
    train = active_metrics(bt, bt.index < TRAIN_END)
    validation = active_metrics(bt, (bt.index >= TRAIN_END) & (bt.index < OOS_START))
    oos = active_metrics(bt, bt.index >= OOS_START)
    full = active_metrics(bt)
    return {
        "strategy": label,
        "mode": "long_short",
        "train_sharpe": train["sharpe"],
        "validation_sharpe": validation["sharpe"],
        "oos_sharpe": oos["sharpe"],
        "oos_pnl": oos["total_pnl"],
        "oos_dd_pct": oos["max_drawdown"] / DD_CAPITAL_USD * 100.0,
        "full_sharpe": full["sharpe"],
        "turnover": full["turnover"],
    }


def zscore_from_train(x_train, x_row):
    mean = x_train.mean()
    std = x_train.std().replace(0.0, np.nan)
    return ((x_train - mean) / std).clip(-5.0, 5.0).fillna(0.0), ((x_row - mean) / std).clip(-5.0, 5.0).fillna(0.0)


def expanding_ols_prediction(x, y, min_train_days=504, refit_every=21):
    preds = pd.Series(np.nan, index=x.index)
    beta, last_fit = None, None
    for i, date in enumerate(x.index):
        train_mask = (x.index < date) & y.notna()
        if train_mask.sum() < min_train_days:
            continue
        if beta is None or last_fit is None or (i - last_fit) >= refit_every:
            x_train, x_row = zscore_from_train(x.loc[train_mask], x.loc[date])
            design = np.column_stack([np.ones(len(x_train)), x_train.values])
            beta, *_ = np.linalg.lstsq(design, y.loc[train_mask].values.astype(float), rcond=None)
            last_fit = i
        else:
            _, x_row = zscore_from_train(x.loc[train_mask], x.loc[date])
        preds.loc[date] = np.r_[1.0, x_row.values.astype(float)].dot(beta)
    return preds


def kalman_prediction(x, y, min_train_days=504, process_noise=1.0e-5):
    columns = list(x.columns)
    beta = np.zeros(len(columns) + 1)
    covariance = np.eye(len(beta)) * 10.0
    mean = pd.Series(0.0, index=columns)
    var = pd.Series(1.0, index=columns)
    target_var = 1.0
    preds = pd.Series(np.nan, index=x.index)
    n = 0
    for date in x.index:
        row = x.loc[date]
        if n > min_train_days:
            z = ((row - mean) / np.sqrt(var.clip(lower=1.0e-8))).clip(-5.0, 5.0)
            preds.loc[date] = np.r_[1.0, z.values.astype(float)].dot(beta)
        y_value = y.loc[date]
        if pd.notnull(y_value):
            n += 1
            old_mean = mean.copy()
            mean = mean + (row - mean) / float(n)
            var = ((n - 2.0) / max(n - 1.0, 1.0)) * var + ((row - old_mean) * (row - mean)) / max(n - 1.0, 1.0)
            target_var = target_var + (float(y_value) ** 2 - target_var) / float(n)
            if n > min_train_days:
                z = ((row - mean) / np.sqrt(var.clip(lower=1.0e-8))).clip(-5.0, 5.0)
                phi = np.r_[1.0, z.values.astype(float)]
                covariance = covariance + np.eye(len(beta)) * float(process_noise)
                innovation_var = float(phi.dot(covariance).dot(phi) + max(target_var, 1.0))
                gain = covariance.dot(phi) / innovation_var
                beta = beta + gain * float(y_value - phi.dot(beta))
                covariance = covariance - np.outer(gain, phi).dot(covariance)
    return preds


def prediction_to_signal(prediction, target_index):
    mean = prediction.rolling(252, min_periods=60).mean().shift(1)
    std = prediction.rolling(252, min_periods=60).std().shift(1).replace(0.0, np.nan)
    return clean_signal((prediction - mean) / std, target_index)


def select_by_ic_signal(families, pnl, target_index, lookback=504):
    target = pnl.shift(-1)
    family_signals = {name: clean_signal(pd.concat(members, axis=1).mean(axis=1), target_index) for name, members in families.items()}
    out = pd.Series(0.0, index=target_index)
    for date in target_index:
        train = target_index < date
        recent = target.loc[train].tail(lookback)
        if recent.notna().sum() < 120:
            continue
        scores = {}
        for name, signal in family_signals.items():
            aligned = pd.concat([signal.loc[recent.index], recent], axis=1).dropna()
            scores[name] = aligned.iloc[:, 0].corr(aligned.iloc[:, 1]) if len(aligned) > 60 else np.nan
        scores = pd.Series(scores).dropna()
        if scores.empty:
            continue
        out.loc[date] = family_signals[scores.abs().idxmax()].loc[date]
    return clean_signal(out, target_index)


def best_family_by_trend_mr(families, panel, target_index):
    trend_strength = panel["mom_60"].abs()
    threshold = trend_strength.expanding(min_periods=252).median().shift(1)
    trend_regime = (trend_strength > threshold).fillna(False)
    price_trend = pd.concat(families["price_trend"], axis=1).mean(axis=1)
    price_mr = pd.concat(families["price_mr"], axis=1).mean(axis=1)
    return clean_signal(price_trend.where(trend_regime, price_mr), target_index)


def named_period_check(bt):
    rows = []
    for item in REGIME_PERIODS:
        mask = (bt.index >= pd.Timestamp(item["start"])) & (bt.index <= pd.Timestamp(item["end"]))
        m = active_metrics(bt, mask)
        rows.append({
            "period": item["period"],
            "total_pnl": m["total_pnl"],
            "sharpe": m["sharpe"],
            "max_dd_pct": m["max_drawdown"] / DD_CAPITAL_USD * 100.0 if pd.notnull(m["max_drawdown"]) else np.nan,
            "hit_rate": m["hit_rate"],
            "days": m["days"],
        })
    return pd.DataFrame(rows)


WHEAT = ["WHEAT_SRW", "WHEAT_HRW"]


def pair_components(panel):
    return {
        "price_trend": clean_signal((panel["mom_20"] + panel["mom_60"]) / 2.0, panel.index),
        "price_mr": clean_signal(panel["rev_5"], panel.index),
        "curve": clean_signal((panel["curve_spread"] + panel["curve_ratio"] + panel["curve_change_20"]) / 3.0, panel.index),
        "cot": clean_signal((panel["cot_mm_level"] + panel["cot_mm_change"] + panel["cot_pm_oi_level"] + panel["cot_pm_oi_change"]) / 4.0, panel.index),
        "physical_public": clean_signal((-panel["public_inventory_change"] - panel["receipts_change"]) / 2.0, panel.index),
        "physical_cargill": clean_signal((-panel["cgl_inventory_change"] + panel["crush_surprise"] + panel["crush_utilization"]) / 3.0, panel.index),
    }


def pair_signal(component_name):
    return clean_signal(srw_components[component_name] - hrw_components[component_name], trading_index)


def wheat_pair_positions(signal, pnl, target_daily_pair_vol=40.0, max_leg_lot=0.40, signal_threshold=0.12, halflife=5.0, rebalance_every=5):
    signal = clean_signal(signal, pnl.index)
    signal = pd.Series(np.tanh(signal / 2.0), index=pnl.index).ewm(halflife=halflife, adjust=False, min_periods=1).mean()
    signal[signal.abs() < signal_threshold] = 0.0
    vol = pnl[WHEAT].rolling(60, min_periods=20).std().shift(1).replace(0.0, np.nan)
    positions = pd.DataFrame(0.0, index=pnl.index, columns=WHEAT)
    positions["WHEAT_SRW"] = (signal * target_daily_pair_vol / vol["WHEAT_SRW"]).clip(-max_leg_lot, max_leg_lot)
    positions["WHEAT_HRW"] = (-signal * target_daily_pair_vol / vol["WHEAT_HRW"]).clip(-max_leg_lot, max_leg_lot)
    positions = positions.fillna(0.0)
    if rebalance_every > 1:
        rebalance_mask = pd.Series(False, index=positions.index)
        rebalance_mask.iloc[::rebalance_every] = True
        positions = positions.where(rebalance_mask).ffill().fillna(0.0)
    return positions


def backtest_pair(signal):
    return backtest_positions_with_costs(
        wheat_pair_positions(signal, futures_pnl),
        futures_pnl[WHEAT],
        trade_cost_per_lot=TRADE_COST_PER_LOT,
        holding_cost_rate=HOLDING_COST_RATE,
        margin_per_lot=DEFAULT_MARGIN_PER_LOT,
    )[0]


def pair_metric_row(label, bt):
    row = metric_row(label, bt)
    row["book"] = "SRW_HRW_PAIR"
    return row


## 3. Generic Signal A/B Tests

In [4]:
srw_components = pair_components(srw_panel)
hrw_components = pair_components(hrw_panel)
pair_families_A = {
    "price_trend": [pair_signal("price_trend")],
    "price_mr": [pair_signal("price_mr")],
    "curve": [pair_signal("curve")],
    "cot": [pair_signal("cot")],
}
pair_families_B = {
    **pair_families_A,
    "physical_public": [pair_signal("physical_public")],
    "physical_cargill": [pair_signal("physical_cargill")],
}
pair_pnl_proxy = futures_pnl["WHEAT_SRW"] - futures_pnl["WHEAT_HRW"]

generic_rows = []
for signal_set, families in {"A": pair_families_A, "B": pair_families_B}.items():
    family_features = pd.DataFrame({name: pd.concat(members, axis=1).mean(axis=1) for name, members in families.items()}, index=trading_index).fillna(0.0)
    target = pair_pnl_proxy.shift(-1)
    tests = {
        "avg_all_signals": family_average(families, trading_index),
        "equal_family": equal_family(families, trading_index),
        "best_family_by_trend_mr": best_family_by_trend_mr(families, pd.DataFrame({"mom_60": family_features.get("price_trend", 0.0)}, index=trading_index), trading_index),
        "select_by_ic": select_by_ic_signal(families, target, trading_index),
        "expanding_ols_family_model": prediction_to_signal(expanding_ols_prediction(family_features, target), trading_index),
        "kalman_family_model": prediction_to_signal(kalman_prediction(family_features, target), trading_index),
    }
    for name, signal in tests.items():
        row = {"signal_set": signal_set}
        row.update(pair_metric_row(name, backtest_pair(signal)))
        generic_rows.append(row)

generic_results = pd.DataFrame(generic_rows).sort_values(["signal_set", "validation_sharpe", "oos_sharpe"], ascending=[True, False, False])
display(generic_results.round(3))

,signal_set,strategy,mode,train_sharpe,validation_sharpe,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,turnover,book
4,A,expanding_ols_family_model,long_short,0.332,0.576,-0.408,-117.106,-2.043,0.059,0.004,SRW_HRW_PAIR
3,A,select_by_ic,long_short,1.058,-1.563,0.030,2.017,-0.728,-0.236,0.004,SRW_HRW_PAIR
5,A,kalman_family_model,long_short,0.054,-1.807,-1.170,-260.384,-3.175,-0.729,0.005,SRW_HRW_PAIR
2,A,best_family_by_trend_mr,long_short,-0.027,-2.157,-0.342,-40.161,-1.002,-0.638,0.003,SRW_HRW_PAIR
0,A,avg_all_signals,long_short,0.715,-2.573,0.371,11.199,-0.361,0.282,0.002,SRW_HRW_PAIR
1,A,equal_family,long_short,0.715,-2.573,0.371,11.199,-0.361,0.282,0.002,SRW_HRW_PAIR
10,B,expanding_ols_family_model,long_short,0.007,-0.030,-0.359,-91.268,-1.781,-0.153,0.005,SRW_HRW_PAIR
9,B,select_by_ic,long_short,0.934,-1.979,0.809,29.942,-0.345,0.187,0.004,SRW_HRW_PAIR
8,B,best_family_by_trend_mr,long_short,-0.027,-2.157,-0.342,-40.161,-1.002,-0.638,0.003,SRW_HRW_PAIR
11,B,kalman_family_model,long_short,-0.043,-2.210,-1.200,-247.256,-3.049,-0.854,0.005,SRW_HRW_PAIR


## 4. Fixed Wheat Pair Benchmark

In [5]:
pair_price_mr = pair_signal("price_mr")
pair_physical_cargill = pair_signal("physical_cargill")
price = data["adj1"].reindex(trading_index).ffill()
ratio = (price["WHEAT_SRW"] / price["WHEAT_HRW"]).replace([np.inf, -np.inf], np.nan).ffill()
trend_20 = rolling_zscore(ratio.pct_change(20), 252, 60).reindex(trading_index).fillna(0.0)
trend_60 = rolling_zscore(ratio.pct_change(60), 252, 80).reindex(trading_index).fillna(0.0)
pair_price_trend = clean_signal((trend_20 + trend_60) / 2.0, trading_index)

mr_cargill_90_10 = clean_signal(0.90 * pair_price_mr + 0.10 * pair_physical_cargill, trading_index)
trend_strength = pair_price_trend.abs()
trend_state = (trend_strength > trend_strength.expanding(min_periods=252).median().shift(1)).fillna(False)
conflict = trend_state & ((mr_cargill_90_10 * pair_price_trend) < 0.0)
trend_conflict_flat = clean_signal(mr_cargill_90_10.where(~conflict, 0.0), trading_index)

pair_candidates = {
    "pair_price_mr_cost_control": pair_price_mr,
    "pair_price_mr_cargill_90_10_cost_control": mr_cargill_90_10,
    "pair_price_mr_cargill_80_20_pair_trend_cost_control": clean_signal(0.80 * mr_cargill_90_10 + 0.20 * pair_price_trend, trading_index),
    "pair_price_mr_cargill_trend_conflict_flat_cost_control": trend_conflict_flat,
}

pair_rows = [pair_metric_row(name, backtest_pair(signal)) for name, signal in pair_candidates.items()]
display(pd.DataFrame(pair_rows).sort_values("oos_sharpe", ascending=False).round(3))

,strategy,mode,train_sharpe,validation_sharpe,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,turnover,book
3,pair_price_mr_cargill_trend_conflict_flat_cost...,long_short,1.980,0.301,2.145,26.671,-0.170,1.597,0.005,SRW_HRW_PAIR
2,pair_price_mr_cargill_80_20_pair_trend_cost_co...,long_short,2.075,-0.928,1.291,36.374,-0.294,1.243,0.003,SRW_HRW_PAIR
1,pair_price_mr_cargill_90_10_cost_control,long_short,0.406,3.045,1.129,26.689,-0.199,1.258,0.005,SRW_HRW_PAIR
0,pair_price_mr_cost_control,long_short,1.076,2.795,1.035,32.965,-0.283,1.376,0.005,SRW_HRW_PAIR


## 5. Annual Walk-Forward Pair Selection

In [6]:
pair_backtests = {name: backtest_pair(signal) for name, signal in pair_candidates.items()}
walk_forward_bt = None
wf_rows = []
for year in range(OOS_START.year, trading_index.max().year + 1):
    start = pd.Timestamp(f"{year}-01-01")
    end = pd.Timestamp(f"{year + 1}-01-01")
    validation_start = start - pd.DateOffset(years=2)
    train_mask = trading_index < validation_start
    validation_mask = (trading_index >= validation_start) & (trading_index < start)
    trade_mask = (trading_index >= start) & (trading_index < end)
    scored = []
    for name, bt in pair_backtests.items():
        train = active_metrics(bt, train_mask)
        validation = active_metrics(bt, validation_mask)
        scored.append({"year": year, "candidate": name, "train_sharpe": train["sharpe"], "validation_sharpe": validation["sharpe"], "score": validation["sharpe"] if train["sharpe"] > 0 else -np.inf})
    selected = pd.DataFrame(scored).sort_values(["score", "validation_sharpe"], ascending=False).iloc[0]["candidate"]
    if walk_forward_bt is None:
        walk_forward_bt = pair_backtests[selected].copy() * 0.0
    walk_forward_bt.loc[trade_mask, pair_backtests[selected].columns] = pair_backtests[selected].loc[trade_mask]
    trade = active_metrics(pair_backtests[selected], trade_mask)
    wf_rows.append({"year": year, "selected": selected, "trade_sharpe": trade["sharpe"], "trade_pnl": trade["total_pnl"], "trade_dd_pct": trade["max_drawdown"] / DD_CAPITAL_USD * 100.0, "active_days": trade["days"]})
walk_forward_bt["cum_pnl"] = walk_forward_bt["net_pnl"].cumsum()

display(pd.DataFrame(wf_rows).round(3))
display(pd.DataFrame([pair_metric_row("annual_walk_forward_pair_selection", walk_forward_bt)]).round(3))

,year,selected,trade_sharpe,trade_pnl,trade_dd_pct,active_days
0,2018,pair_price_mr_cargill_90_10_cost_control,10.126,32.947,-0.032,25.000
1,2019,pair_price_mr_cargill_trend_conflict_flat_cost...,0.711,3.941,-0.153,35.000
2,2020,pair_price_mr_cargill_trend_conflict_flat_cost...,1.166,6.498,-0.170,30.000


,strategy,mode,train_sharpe,validation_sharpe,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,turnover,book
0,annual_walk_forward_pair_selection,long_short,NaN,NaN,2.963,43.385,-0.170,2.963,0.005,SRW_HRW_PAIR


## 6. Final Wheat Candidate

In [7]:
FINAL_NAME = "pair_price_mr_cargill_trend_conflict_flat_cost_control"
final_bt = pair_backtests[FINAL_NAME]
final_row = pd.DataFrame([pair_metric_row(FINAL_NAME, final_bt)])
display(final_row.round(3))

,strategy,mode,train_sharpe,validation_sharpe,oos_sharpe,oos_pnl,oos_dd_pct,full_sharpe,turnover,book
0,pair_price_mr_cargill_trend_conflict_flat_cost...,long_short,1.980,0.301,2.145,26.671,-0.170,1.597,0.005,SRW_HRW_PAIR


## 7. Named-period Check

In [8]:
period_check = named_period_check(final_bt)
display(period_check.round(3))
row = final_row.iloc[0]
display(Markdown(f"""
## Conclusion

Final wheat candidate: `{FINAL_NAME}`.

OOS Sharpe `{row['oos_sharpe']:.3f}`, OOS PnL `{row['oos_pnl']:.3f}`, OOS max DD `{row['oos_dd_pct']:.3f}%`.

Wheat survives best as an SRW/HRW pair sleeve. The generic A/B tests are retained as diagnostics, while the promoted row uses a fixed price-MR spine, a small Cargill physical overlay, and a trend-conflict flat rule.
"""))

,period,total_pnl,sharpe,max_dd_pct,hit_rate,days
0,Russian drought/export ban shock,9.069,7.638,-0.020,0.600,10.000
1,US drought rally/retrace,3.334,4.519,-0.029,0.600,5.000
2,Crimea/Black Sea shock,8.259,6.412,-0.050,0.700,10.000
3,Low-price abundant supply,21.731,1.685,-0.167,0.488,80.000
4,US-China trade war,3.941,0.711,-0.153,0.457,35.000
5,2019 prevented planting floods,-11.752,-5.150,-0.153,0.333,15.000
6,COVID demand shock,5.114,1.171,-0.170,0.500,20.000
7,COVID recovery/China buying,2.056,3.334,-0.032,0.600,5.000



## Conclusion

Final wheat candidate: `pair_price_mr_cargill_trend_conflict_flat_cost_control`.

OOS Sharpe `2.145`, OOS PnL `26.671`, OOS max DD `-0.170%`.

Wheat survives best as an SRW/HRW pair sleeve. The generic A/B tests are retained as diagnostics, while the promoted row uses a fixed price-MR spine, a small Cargill physical overlay, and a trend-conflict flat rule.
